# Notebook 39 — Local Failure and Claim-Boundary Audit

Accepts NB38 `NO_GO`. Sealed stays closed.

## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    PROJECT_ROOT = NOTEBOOK_DIR
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (parent / "S4_sbi" / "src" / "sleep_sbi").is_dir():
            PROJECT_ROOT = parent
            break
SRC = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sleep_sbi.figure10_closure_common import load_scientific_protocol, SCI_LOCK_PATH
proto = load_scientific_protocol()
display(Markdown(f"Scientific protocol: `{SCI_LOCK_PATH}`"))
display(pd.Series({
    'global_uncertainty_status': proto['global_uncertainty_status'],
    'intervention_frozen': proto['intervention_status_frozen'],
    'e2_ref_7d': proto['e2_ref']['7d'],
    'e2_ref_8d': proto['e2_ref']['8d'],
}))


Scientific protocol: `D:\Year3_Mao_Projects\sleep_loop\S4_sbi\configs\notebook_39_41_closure\scientific_protocol_lock.json`

global_uncertainty_status                    NO_GO
intervention_frozen          CONTROL_NOT_JUSTIFIED
e2_ref_7d                                 member_4
e2_ref_8d                                 member_3
dtype: object

## 2. Lineage audit + failure ledger + support conflict

In [2]:
from sleep_sbi.figure10_notebook39_claim_boundary import (
    audit_metric_lineage, build_parameter_failure_ledger, resolve_support_conflict, run_notebook39
)
lineage = audit_metric_lineage()
display(Markdown('### Metric lineage'))
display(pd.DataFrame([
    {'metric': k, 'class': v.get('class')} for k, v in lineage['metrics'].items()
]))
ledger = build_parameter_failure_ledger()
display(Markdown('### Failure ledger (head)'))
display(ledger.head(20))
support = resolve_support_conflict()
display(Markdown(f"### Support resolution: **{support['resolution_label']}**"))
display(Markdown(support['failure_reason_wording']))


### Metric lineage

,metric,class
0,so_waveform,decision_held_out_descriptive
1,spindle_density,decision_held_out_descriptive
2,figure10_rate14_ppc,development_exposed_synthetic
3,shape_r_T1_T12_PAC,fitting_or_constraint_derived


### Failure ledger (head)

,track,estimator,parameter,family,statistic,p_value,rank_mean,holm_clear_issue,tag,classification,note
0,7d,member_4,mue,marginal_sbc,0.084082,9.549584e-07,0.522927,True,formal_diagnostic,NaN,NaN
1,7d,member_4,mui,marginal_sbc,0.288300,7.509242e-76,0.335821,True,formal_diagnostic,NaN,NaN
2,7d,member_4,b,marginal_sbc,0.045446,2.821325e-02,0.483955,False,formal_diagnostic,NaN,NaN
3,7d,member_4,tauA,marginal_sbc,0.071858,4.822451e-05,0.516959,True,formal_diagnostic,NaN,NaN
4,7d,member_4,g_LK,marginal_sbc,0.034972,1.595204e-01,0.500482,False,formal_diagnostic,NaN,NaN
5,7d,member_4,g_h,marginal_sbc,0.035135,1.558194e-01,0.507256,False,formal_diagnostic,NaN,NaN
6,7d,member_4,c_th2ctx,marginal_sbc,0.049396,1.305658e-02,0.507233,False,formal_diagnostic,NaN,NaN
7,7d,member_4,mue,coverage_90,0.806641,NaN,NaN,True,formal_diagnostic,unhealthy_under,NaN
8,7d,member_4,mui,coverage_90,0.931641,NaN,NaN,False,formal_diagnostic,healthy,NaN
9,7d,member_4,b,coverage_90,0.862305,NaN,NaN,True,formal_diagnostic,crossing,NaN


### Support resolution: **sample_set_difference**

Cursor NB38 support QA and Claude NB38 reproducibility audit operate on different sample sets and boundary definitions (exact atoms vs near-boundary tol=1e-05). No shared sample-ID alignment artifacts were available to escalate beyond sample_set_difference. NB38 global NO_GO remains immutable (S_fail/C_fail|C_lim/L_fail).

## 3. Persist summary / NB40 handoff

In [3]:
summary = run_notebook39()
display(pd.Series(summary))
assert summary['global_uncertainty_status'] == 'NO_GO'
assert summary['nb40_handoff_status'] == 'CONTINUE'
display(Markdown('**STOP after NB39 only if integrity failed; otherwise continue to NB40.**'))


created_utc                                     2026-08-09T14:50:50.085136+00:00
global_uncertainty_status                                                  NO_GO
local_analysis_mode                                       negative_evidence_only
e2_ref                                      {'7d': 'member_4', '8d': 'member_3'}
support_conflict_resolution                                sample_set_difference
failure_reason_wording         Cursor NB38 support QA and Claude NB38 reprodu...
decision_held_out_count                                                        2
nb40_handoff_status                                                     CONTINUE
nb38_decision_sha256           c67ea6c3d110cd1bacf20020050c93a93a7375eb699b0b...
scientific_protocol            S4_sbi/configs/notebook_39_41_closure/scientif...
dtype: object

**STOP after NB39 only if integrity failed; otherwise continue to NB40.**